In [1]:
import numpy as np
import pandas as pd

In [5]:
# data_ids = [361260, 361254, 361259, 361253, 361243, 361242]
data_ids = [361254, 361259, 361253, 361243, 361242]
n_ests = [50, 100, 500, 1000]
min_samples_leafs = [1, 5, 10]
max_features = [0.1, 0.33, "1.0"]

In [6]:
# for each data_id, load the result and save as a df
dfs = []
for data_id in data_ids:
    # get number of samples in the data_id by reading X csv
    X = np.loadtxt(f"data/{data_id}/X.csv", delimiter=",")
    n_samples = 1000
    n_features = X.shape[1]
    for n_est in n_ests:
        for min_samples_leaf in min_samples_leafs:
            for max_feature in max_features:
                # create the directory if it doesn't exist
                dir_path = f"results/rf/{data_id}/n_estimators_{n_est}/min_samples_leaf_{min_samples_leaf}/max_features_{max_feature}"
                results_path = f"{dir_path}/runtime_results.csv"
                results_df = pd.read_csv(results_path)
                # divide every col in df except 'data_id' by n_samples
                for col in results_df.columns:
                    if col != 'data_id':
                        results_df[col] = results_df[col] / n_samples
                # add columns for n_estimators, min_samples_leaf, max_features
                results_df['n_estimators'] = n_est
                results_df['min_samples_leaf'] = min_samples_leaf
                results_df['max_features'] = max_feature
                results_df['num_features'] = n_features
                dfs.append(results_df)
df = pd.concat(dfs, ignore_index=True)

In [7]:
df[(df['max_features'] == 0.33) & (df['n_estimators'] == 100)].sort_values(by=['min_samples_leaf', 'data_id'])

,data_id,rf_fitting_time,rf_plus_fitting_time,shap_rf_explainer_time,shap_rf_values_time,lime_rf_time,local_mdi_time,lmdi_plus_rf_explainer_time,lmdi_plus_rf_values_time,n_estimators,min_samples_leaf,max_features,num_features
154,361242,0.001498,0.047808,0.000030,0.031717,0.186433,0.001082,0.000006,0.043541,100,1,0.33,81
118,361243,0.003104,0.031660,0.000016,0.036066,0.319966,0.001455,0.000014,0.110917,100,1,0.33,116
82,361253,0.001102,0.027166,0.000032,0.033963,0.138509,0.001114,0.000006,0.022529,100,1,0.33,48
10,361254,0.000665,0.022531,0.000025,0.031032,0.126291,0.001288,0.000009,0.015159,100,1,0.33,21
46,361259,0.000813,0.045874,0.000033,0.030291,0.114102,0.001223,0.000006,0.014599,100,1,0.33,32
157,361242,0.001003,0.007108,0.000004,0.003114,0.188586,0.000834,0.000006,0.034292,100,5,0.33,81
121,361243,0.001941,0.009503,0.000010,0.004543,0.303333,0.000926,0.000006,0.049521,100,5,0.33,116
85,361253,0.000853,0.008118,0.000010,0.003883,0.168553,0.000920,0.000006,0.020327,100,5,0.33,48
13,361254,0.000395,0.005046,0.000012,0.002404,0.073060,0.000779,0.000005,0.003176,100,5,0.33,21
49,361259,0.000571,0.005350,0.000012,0.002810,0.095760,0.000815,0.000004,0.005758,100,5,0.33,32


In [9]:
display_df = df[(df['max_features'] == 0.33) & (df['n_estimators'] == 100)].sort_values(by=['min_samples_leaf', 'data_id'])
# display_df columns should be data_id, n_features, n_estimators, min_samples_leaf, max_features, lime_time, shap_values_time, rf_plus_fitting_time + lmdi_plus_values_time
display_df = display_df[['data_id', 'num_features', 'min_samples_leaf', 'lime_rf_time', 'shap_rf_values_time', 'rf_plus_fitting_time', 'lmdi_plus_rf_values_time']]
display_df['lmdi_plus_time'] = display_df['rf_plus_fitting_time'] + display_df['lmdi_plus_rf_values_time']
display_df.drop(columns=['rf_plus_fitting_time', 'lmdi_plus_rf_values_time'], inplace=True)
display_df = display_df.rename(columns={
    'data_id': 'OpenML Data ID',
    'num_features': '# of Features',
    'n_estimators': '# of Estimators',
    'min_samples_leaf': 'Min. Samples per Leaf',
    'max_features': 'Max Features per Split',
    'lime_time': 'LIME',
    'shap_values_time': 'TreeSHAP',
    'lmdi_plus_time': 'LMDI+'
})
display_df

,OpenML Data ID,# of Features,Min. Samples per Leaf,lime_rf_time,shap_rf_values_time,LMDI+
154,361242,81,1,0.186433,0.031717,0.091349
118,361243,116,1,0.319966,0.036066,0.142577
82,361253,48,1,0.138509,0.033963,0.049695
10,361254,21,1,0.126291,0.031032,0.037690
46,361259,32,1,0.114102,0.030291,0.060473
157,361242,81,5,0.188586,0.003114,0.041400
121,361243,116,5,0.303333,0.004543,0.059024
85,361253,48,5,0.168553,0.003883,0.028445
13,361254,21,5,0.073060,0.002404,0.008222
49,361259,32,5,0.095760,0.002810,0.011108


In [10]:
# round to fourth decimal place
display_df = display_df.round(4)

In [11]:
# get display_df in markdown format
markdown_df = display_df.to_markdown(index=False)
print(markdown_df)

|   OpenML Data ID |   # of Features |   Min. Samples per Leaf |   lime_rf_time |   shap_rf_values_time |   LMDI+ |
|-----------------:|----------------:|------------------------:|---------------:|----------------------:|--------:|
|           361242 |              81 |                       1 |         0.1864 |                0.0317 |  0.0913 |
|           361243 |             116 |                       1 |         0.32   |                0.0361 |  0.1426 |
|           361253 |              48 |                       1 |         0.1385 |                0.034  |  0.0497 |
|           361254 |              21 |                       1 |         0.1263 |                0.031  |  0.0377 |
|           361259 |              32 |                       1 |         0.1141 |                0.0303 |  0.0605 |
|           361242 |              81 |                       5 |         0.1886 |                0.0031 |  0.0414 |
|           361243 |             116 |                       5 |        

In [12]:
df[(df['max_features'] == 0.33) & (df['min_samples_leaf'] == 5)].sort_values(by=['n_estimators', 'data_id'])

,data_id,rf_fitting_time,rf_plus_fitting_time,shap_rf_explainer_time,shap_rf_values_time,lime_rf_time,local_mdi_time,lmdi_plus_rf_explainer_time,lmdi_plus_rf_values_time,n_estimators,min_samples_leaf,max_features,num_features
148,361242,0.000532,0.006824,0.000005,0.001733,0.215546,0.000483,0.000003,0.015815,50,5,0.33,81
112,361243,0.000991,0.006760,0.000005,0.002374,0.281929,0.000522,0.000003,0.027139,50,5,0.33,116
76,361253,0.000476,0.005429,0.000007,0.002090,0.156516,0.000562,0.000004,0.008764,50,5,0.33,48
4,361254,0.000180,0.003806,0.000004,0.001255,0.055506,0.000423,0.000002,0.001566,50,5,0.33,21
40,361259,0.000366,0.005601,0.000005,0.001970,0.117854,0.000586,0.000004,0.004816,50,5,0.33,32
157,361242,0.001003,0.007108,0.000004,0.003114,0.188586,0.000834,0.000006,0.034292,100,5,0.33,81
121,361243,0.001941,0.009503,0.000010,0.004543,0.303333,0.000926,0.000006,0.049521,100,5,0.33,116
85,361253,0.000853,0.008118,0.000010,0.003883,0.168553,0.000920,0.000006,0.020327,100,5,0.33,48
13,361254,0.000395,0.005046,0.000012,0.002404,0.073060,0.000779,0.000005,0.003176,100,5,0.33,21
49,361259,0.000571,0.005350,0.000012,0.002810,0.095760,0.000815,0.000004,0.005758,100,5,0.33,32


In [9]:
display_df = df[(df['max_features'] == 0.33) & (df['min_samples_leaf'] == 5) & (df['n_estimators'] != 50)].sort_values(by=['n_estimators', 'data_id'])
# display_df columns should be data_id, n_features, n_estimators, min_samples_leaf, max_features, lime_time, shap_values_time, rf_plus_fitting_time + lmdi_plus_values_time
display_df = display_df[['data_id', 'num_features', 'n_estimators', 'lime_time', 'shap_values_time', 'rf_plus_fitting_time', 'lmdi_plus_values_time']]
display_df['lmdi_plus_time'] = display_df['rf_plus_fitting_time'] + display_df['lmdi_plus_values_time']
display_df.drop(columns=['rf_plus_fitting_time', 'lmdi_plus_values_time'], inplace=True)
display_df = display_df.rename(columns={
    'data_id': 'OpenML Data ID',
    'num_features': '# of Features',
    'n_estimators': '# of Estimators',
    'min_samples_leaf': 'Min. Samples per Leaf',
    'max_features': 'Max Features per Split',
    'lime_time': 'LIME',
    'shap_values_time': 'TreeSHAP',
    'lmdi_plus_time': 'LMDI+'
})
display_df

,OpenML Data ID,# of Features,# of Estimators,LIME,TreeSHAP,LMDI+
193,361242,81,100,0.224189,0.003786,0.041968
157,361243,116,100,0.309116,0.003834,0.073164
121,361253,48,100,0.184714,0.004231,0.025178
49,361254,21,100,0.112093,0.003311,0.018038
85,361259,32,100,0.134084,0.003485,0.020884
13,361260,15,100,0.092258,0.003339,0.013281
202,361242,81,500,0.184918,0.002991,0.035427
166,361243,116,500,0.273784,0.003884,0.063366
130,361253,48,500,0.148719,0.003174,0.020331
58,361254,21,500,0.090191,0.002407,0.011842


In [10]:
# round to fourth decimal place
display_df = display_df.round(4)

In [11]:
# get display_df in markdown format
markdown_df = display_df.to_markdown(index=False)
print(markdown_df)

|   OpenML Data ID |   # of Features |   # of Estimators |   LIME |   TreeSHAP |   LMDI+ |
|-----------------:|----------------:|------------------:|-------:|-----------:|--------:|
|           361242 |              81 |               100 | 0.2242 |     0.0038 |  0.042  |
|           361243 |             116 |               100 | 0.3091 |     0.0038 |  0.0732 |
|           361253 |              48 |               100 | 0.1847 |     0.0042 |  0.0252 |
|           361254 |              21 |               100 | 0.1121 |     0.0033 |  0.018  |
|           361259 |              32 |               100 | 0.1341 |     0.0035 |  0.0209 |
|           361260 |              15 |               100 | 0.0923 |     0.0033 |  0.0133 |
|           361242 |              81 |               500 | 0.1849 |     0.003  |  0.0354 |
|           361243 |             116 |               500 | 0.2738 |     0.0039 |  0.0634 |
|           361253 |              48 |               500 | 0.1487 |     0.0032 |  0.0203 |